# Primeros pasos con `dane-data`

Este notebook recorre la librería con los microdatos del DANE:

1. Configurar carpetas y descargar
2. GEIH: cargar, indicadores laborales, tablas ponderadas, brechas
3. EAM: resumen sectorial, productividad, panel de establecimientos

> Si aún no tiene datos, descomente las celdas de descarga.

In [ ]:
import dane_data as dd
import pandas as pd

dd.show_paths()   # carpetas configuradas (cámbielas con dd.set_paths(..., save=True))

In [ ]:
# Descarga (incremental). Descomente si necesita los datos:
# dd.geih.download([2024, 2025])
# dd.eam.download(range(2014, 2025))

dd.geih.available().groupby("anio").size().tail()

## 1. GEIH: una persona por fila

`load` une los módulos (características, fuerza de trabajo, ocupados, no ocupados, hogar) y agrega
variables **armonizadas en minúscula**, iguales en todos los años.

In [ ]:
df = dd.geih.load(2024)
df[["anio", "mes", "fex", "sexo", "edad", "condicion", "formal", "ingreso_laboral", "ratio_smlv", "nivel_educativo"]].head()

## 2. Indicadores del mercado laboral

Niveles en personas (promedio mensual) y tasas en %. Compare con el boletín del DANE.
Nota: en los datos locales falta *No ocupados* de marzo 2024, así que ese mes se omite (verá un aviso).

In [ ]:
dd.geih.indicadores(df).round(1)

In [ ]:
dd.geih.indicadores(df, por="sexo")[["sexo", "tgp", "to", "td", "tasa_informalidad", "ingreso_mediano"]].round(1)

In [ ]:
# Desempleo por grupo de edad, mes a mes
edad = dd.geih.indicadores(df, por="grupo_edad", anual=False)
edad.pivot(index="mes", columns="grupo_edad", values="td").round(1)

## 3. Tablas ponderadas de cualquier variable

`dd.tabla` usa el factor de expansión y divide por el número de meses automáticamente.

In [ ]:
ocupados = df[df.ocupado == 1]
dd.tabla(ocupados, por=["nivel_educativo"], variable="ingreso_laboral", estadistico="mediana")

In [ ]:
# ¿Qué porcentaje de los ocupados gana alrededor de un salario mínimo?
w = dd.stats.peso_mensual_promedio(ocupados)
cerca = ocupados.ratio_smlv.between(0.95, 1.05)
print(f"{100 * w[cerca].sum() / w.sum():.1f} % de los ocupados gana ≈ 1 SMLV")

In [ ]:
# Variables del DANE que no están en el núcleo: búsquelas y cárguelas con columns=[...]
dd.buscar_variable("arriendo").head()

## 4. EAM: industria manufacturera

Un establecimiento-año por fila. Los montos están en **miles de pesos corrientes**.

In [ ]:
eam = dd.eam.load(range(2014, 2025))
res = dd.eam.resumen(eam, por="anio")
res[["anio", "establecimientos", "personal_total", "valor_agregado", "productividad_laboral", "participacion_laboral"]].round(1)

In [ ]:
# Sectores con mayor valor agregado en 2024
(dd.eam.resumen(eam[eam.anio == 2024], por="sector")
   .sort_values("valor_agregado", ascending=False)
   .head(8)[["sector", "establecimientos", "valor_agregado", "productividad_laboral"]])

In [ ]:
# Panel balanceado de plantas, listo para econometría (p. ej. linearmodels.PanelOLS)
panel = dd.eam.panel(eam, balanceado=True)
print(panel.index.get_level_values("nordest").nunique(), "establecimientos presentes todos los años")
panel[["valagri", "pertotal", "activfi"]].head()

## 5. Exportar

Todo es un `pandas.DataFrame`: `to_csv`, `to_excel`, `to_stata`, `to_parquet`…

```python
dd.geih.indicadores(df, por="departamento").to_excel("indicadores_dpto.xlsx", index=False)
```